In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = 'https://raw.githubusercontent.com/rwalden-gif/IntroToML/main/D3.csv'
data = pd.read_csv(url)

X1 = data["X1"].values
X2 = data["X2"].values
X3 = data["X3"].values
Y  = data["Y"].values
m = len(Y)

print(data.head())
print("Number of samples:", m)


In [ ]:
def gradient_descent(X, y, alpha=0.05, iterations=1000):
    """
    X: 2D numpy array, shape (m, n_features)  -- WITHOUT bias column, we add it here
    y: 1D numpy array, shape (m,)
    Returns: theta (final params), cost_history (list)
    """
    m = len(y)
    X_b = np.c_[np.ones(m), X]          # add bias/intercept column of 1s
    n = X_b.shape[1]
    theta = np.zeros(n)                  # theta initialized to zero
    cost_history = []

    for it in range(iterations):
        predictions = X_b.dot(theta)
        errors = predictions - y
        gradient = (1/m) * X_b.T.dot(errors)
        theta = theta - alpha * gradient
        cost = (1/(2*m)) * np.sum(errors**2)   # MSE-style cost
        cost_history.append(cost)

    return theta, cost_history


In [ ]:
learning_rates = [0.1, 0.05, 0.01]
iterations = 1000
variables = {"X1": X1, "X2": X2, "X3": X3}

results_p1 = {}  # store best theta/cost per variable

for name, x in variables.items():
    print(f"\n===== {name} =====")
    best_alpha, best_theta, best_cost_hist = None, None, None

    for alpha in learning_rates:
        theta, cost_hist = gradient_descent(x.reshape(-1,1), Y, alpha=alpha, iterations=iterations)
        print(f"alpha={alpha}: final cost={cost_hist[-1]:.5f}, theta0={theta[0]:.4f}, theta1={theta[1]:.4f}")

        # keep the lowest-cost run as the "best" model for this variable
        if best_cost_hist is None or cost_hist[-1] < best_cost_hist[-1]:
            best_alpha, best_theta, best_cost_hist = alpha, theta, cost_hist

    results_p1[name] = (best_alpha, best_theta, best_cost_hist, x)
    print(f">> Best for {name}: alpha={best_alpha}, model: Y = {best_theta[0]:.4f} + {best_theta[1]:.4f}*{name}")


In [ ]:
for name, (alpha, theta, cost_hist, x) in results_p1.items():
    fig, axes = plt.subplots(1, 2, figsize=(12,4))

    # Regression fit plot
    axes[0].scatter(x, Y, color="blue", label="Data")
    x_sorted = np.sort(x)
    y_pred = theta[0] + theta[1]*x_sorted
    axes[0].plot(x_sorted, y_pred, color="red", label="Regression line")
    axes[0].set_xlabel(name)
    axes[0].set_ylabel("Y")
    axes[0].set_title(f"Regression: Y vs {name} (alpha={alpha})")
    axes[0].legend()

    # Loss curve plot
    axes[1].plot(range(len(cost_hist)), cost_hist, color="green")
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel("Cost (Loss)")
    axes[1].set_title(f"Loss over iterations for {name}")

    plt.tight_layout()
    plt.show()

# Compare final losses to answer "which variable explains Y best"
for name, (alpha, theta, cost_hist, x) in results_p1.items():
    print(f"{name}: final loss = {cost_hist[-1]:.5f}")


In [ ]:
for name, x in variables.items():
    plt.figure(figsize=(6,4))
    for alpha in learning_rates:
        _, cost_hist = gradient_descent(x.reshape(-1,1), Y, alpha=alpha, iterations=iterations)
        plt.plot(cost_hist, label=f"alpha={alpha}")
    plt.xlabel("Iteration")
    plt.ylabel("Cost")
    plt.title(f"Effect of learning rate on loss ({name})")
    plt.legend()
    plt.show()


In [ ]:
X_multi = np.column_stack((X1, X2, X3))

learning_rates_p2 = [0.1, 0.05, 0.01]
iterations_p2 = 1000

best_alpha2, best_theta2, best_cost2 = None, None, None

for alpha in learning_rates_p2:
    theta, cost_hist = gradient_descent(X_multi, Y, alpha=alpha, iterations=iterations_p2)
    print(f"alpha={alpha}: final cost={cost_hist[-1]:.5f}, theta={theta}")
    if best_cost2 is None or cost_hist[-1] < best_cost2[-1]:
        best_alpha2, best_theta2, best_cost2 = alpha, theta, cost_hist

print(f"\nBest alpha: {best_alpha2}")
print(f"Final model: Y = {best_theta2[0]:.4f} + {best_theta2[1]:.4f}*X1 + {best_theta2[2]:.4f}*X2 + {best_theta2[3]:.4f}*X3")


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(best_cost2, color="purple")
plt.xlabel("Iteration")
plt.ylabel("Cost (Loss)")
plt.title(f"Loss over iterations — Multivariable model (alpha={best_alpha2})")
plt.show()

# Compare learning rates on one chart
plt.figure(figsize=(6,4))
for alpha in learning_rates_p2:
    _, cost_hist = gradient_descent(X_multi, Y, alpha=alpha, iterations=iterations_p2)
    plt.plot(cost_hist, label=f"alpha={alpha}")
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Effect of learning rate on loss (multivariable model)")
plt.legend()
plt.show()


In [ ]:
theta = best_theta2
new_points = [(1,1,1), (2,0,4), (3,2,1)]

for (x1,x2,x3) in new_points:
    y_pred = theta[0] + theta[1]*x1 + theta[2]*x2 + theta[3]*x3
    print(f"Prediction for (X1={x1}, X2={x2}, X3={x3}): Y = {y_pred:.4f}")
